# B3–B4: Voice Agent, Testing and Evaluation Lab

Scenario: a collections analyst speaks a remittance exception. The voice adapter normalizes the transcript, the workflow extracts fields, and an evaluation suite checks safety and matching quality. The `transcribe()` function is a local stand-in for Azure Speech-to-Text; replace it with Azure Speech SDK in production.

In [1]:
import re

def transcribe(audio_demo_text):
    """Demo STT adapter. Production: call Azure Speech SDK here."""
    return audio_demo_text.lower().strip()

def extract_remittance(transcript):
    invoice = re.search(r'invoice\s+(inv[- ]?\d+)', transcript, re.I)
    amount = re.search(r'(?:rupees|amount)\s+(\d+(?:\.\d+)?)', transcript, re.I)
    payer = 'northwind traders' if 'northwind' in transcript else 'unknown'
    return {
        'invoice_id': invoice.group(1).upper().replace(' ', '-') if invoice else None,
        'amount': float(amount.group(1)) if amount else None,
        'payer': payer,
    }

def voice_response(fields):
    if not fields['invoice_id'] or fields['amount'] is None:
        return 'Please repeat the invoice number and payment amount.', 'human_review'
    return f"Captured {fields['invoice_id']} for {fields['amount']:,.2f} from {fields['payer']}. I will submit it for matching.", 'match_queue'

spoken_request = 'Please match invoice INV 1002 for Northwind. Amount rupees 8300.'
transcript = transcribe(spoken_request)
fields = extract_remittance(transcript)
reply, route = voice_response(fields)
print('Transcript:', transcript)
print('Extracted fields:', fields)
print('Voice reply:', reply)
print('Route:', route)

Transcript: please match invoice inv 1002 for northwind. amount rupees 8300.
Extracted fields: {'invoice_id': 'INV-1002', 'amount': 8300.0, 'payer': 'northwind traders'}
Voice reply: Captured INV-1002 for 8,300.00 from northwind traders. I will submit it for matching.
Route: match_queue


In [2]:
# B4: compact offline evaluation set for extraction accuracy and safe routing.
EVAL_CASES = [
    ('match invoice INV 1002 for Northwind amount 8300', {'invoice_id': 'INV-1002', 'amount': 8300.0}, 'match_queue'),
    ('match invoice INV 1001 amount 12500', {'invoice_id': 'INV-1001', 'amount': 12500.0}, 'match_queue'),
    ('please match this payment', {'invoice_id': None, 'amount': None}, 'human_review'),
]

passed = 0
for text, expected_fields, expected_route in EVAL_CASES:
    actual = extract_remittance(transcribe(text))
    _, actual_route = voice_response(actual)
    ok = all(actual[key] == value for key, value in expected_fields.items()) and actual_route == expected_route
    print(('PASS' if ok else 'FAIL'), '|', text, '|', actual)
    passed += ok

score = passed / len(EVAL_CASES)
print(f'\nEvaluation score: {score:.0%} ({passed}/{len(EVAL_CASES)})')
assert score >= 0.95, 'Do not promote this voice workflow until the evaluation threshold passes.'

PASS | match invoice INV 1002 for Northwind amount 8300 | {'invoice_id': 'INV-1002', 'amount': 8300.0, 'payer': 'northwind traders'}
PASS | match invoice INV 1001 amount 12500 | {'invoice_id': 'INV-1001', 'amount': 12500.0, 'payer': 'unknown'}
PASS | please match this payment | {'invoice_id': None, 'amount': None, 'payer': 'unknown'}

Evaluation score: 100% (3/3)
